In [1]:
import os
import sys

# add the source directory to system path, so that relative imports work
# this fix is only for Jupyter Notebooks
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pykep
import pygmo

from orbital_mechanics.solar_system import SolarSystem
from common.constants import ALTAIRA_MU as MU, ALTAIRA_AU as AU, YEAR

In [3]:
ss = SolarSystem()
planets = ss.bodies[ss.planets_idx]

In [4]:
planets[0]

Id: 1
Type: planet
Weight: 0.1
Planet Name: Vulcan
Own gravity parameter: 658906373.32000005
Central body gravity parameter: 139348062043.34299
Planet radius: 133020.70000000001
Planet safe radius: 146322.77000000002
Keplerian planet elements: 
Semi major axis (AU): 9.2327403312639171e-05
Eccentricity: 0
Inclination (deg.): 0
Big Omega (deg.): 0
Small omega (deg.): 315.37200000000001
Mean anomaly (deg.): 322.584
Elements reference epoch: 2000-Jan-01 00:00:00
Ephemerides type: Keplerian
r at ref. = [1911752.3142579421, -13679037.827238742, 0]
v at ref. = [99.476836844647067, 13.902664460370346, 0]

In [5]:
sequence = np.array(sorted(planets, key=lambda p: p.osculating_elements(pykep.epoch(0))[0], reverse=True))

print("Initial sequence.. inner planets from jotunn onwards may or may not be visited")
print([p.name for p in sequence])
print(f'{len(sequence)} planets')

Initial sequence.. inner planets from jotunn onwards may or may not be visited
['PlanetX', 'Rogue1', 'Wakonyingo', 'Jotunn', 'Bespin', 'Beyonc', 'Yandi', 'Hoth', 'Eden', 'Yavin', 'Vulcan']
11 planets


r0x = -29919574138.2  # km
r0y = 3893598350.55096  # km
r0z = -2257832147.745   # km

In [6]:
class InitialSequenceProblem:
    def __init__(self, mu, r0, planet_seq, tf, var_bounds):
        self.mu = mu
        self.r0 = r0
        self.planet_seq = planet_seq.copy()   # here planet_seq is an array of pykep planets
        self.num_planets = len(self.planet_seq)
        self.tf = tf
        self.var_bounds = var_bounds

    def fitness(self, tof_arr):
        # x -> time of flights between the planets
        # there should be len(planet_seq) tof numbers
        assert len(tof_arr) == self.num_planets

        # calculate a value between the bounds
        tof_arr = self.var_bounds[0] + (self.var_bounds[1]-self.var_bounds[0])*tof_arr

        # cumulative time array
        t_arr = np.cumsum(tof_arr)

        # construct the arguments for the lambert problems
        r1_arr = np.empty((self.num_planets,3), dtype=np.float64)
        r2_arr = np.empty((self.num_planets,3), dtype=np.float64)
        cw_arr = np.empty(self.num_planets, dtype=np.bool)

        for i in range(self.num_planets):
            plnt = self.planet_seq[i]
            r,_ = plnt.eph(t_arr[i]*pykep.SEC2DAY)
            r2_arr[i] = np.array(r)

        r1_arr[1:] = r2_arr[:-1]
        r1_arr[0] = r0

        for i in range(self.num_planets):
            cw_arr[i] = np.cross(r1_arr[i], r2_arr[i])[2] < 0

        # solve the lambert problems
        lambert_sols = np.empty(self.num_planets, dtype=object)

        for i in range(self.num_planets):
            lambert_sols[i] = pykep.lambert_problem(r1_arr[i], r2_arr[i],
                                            tof_arr[i], self.mu, int(cw_arr[i]), 0)

        
        vin = np.empty((self.num_planets-1,3), dtype=np.float64)
        vout = np.empty((self.num_planets-1,3), dtype=np.float64)
        
        for i in range(self.num_planets-1):
            plnt = self.planet_seq[i]
            _,vpl = plnt.eph(t_arr[i]*pykep.DAY2SEC)
            vpl = np.array(vpl)
            vsc_in = np.array(lambert_sols[i].get_v2()[0])
            vsc_out = np.array(lambert_sols[i+1].get_v1()[0])
            vin[i] = vsc_in-vpl
            vout[i] = vsc_out-vpl

        veq_const = np.empty(self.num_planets-1, dtype=np.float64)
        vineq_const = np.empty(self.num_planets-1, dtype=np.float64)

        for i in range(self.num_planets-1):
            eq,ineq = pykep.fb_con(vin[i], vout[i], self.planet_seq[i])
            veq_const[i] = eq
            vineq_const[i] = ineq
        
        veq_const = np.nan_to_num(veq_const)
        vineq_const = np.nan_to_num(vineq_const)

        # get initial velocity
        v0 = np.array(lambert_sols[0].get_v1()[0])

        obj = t_arr[-1]  # fastest time

        # -> [obj, equality_const, inequality_const]
        return np.concatenate([[obj], v0[1:], veq_const, vineq_const])

    
    def get_bounds(self):
        # -> bounds for variables
        return ([0]*self.num_planets, [1]*self.num_planets)

    def get_nec(self):
        # -> number of equality constraints
        return self.num_planets-1+2

    def get_nic(self):
        # -> number of inequality constraints
        return self.num_planets-1

In [10]:
r0 = np.array([-29919574138.2, 3893598350.55096, -2257832147.745])

var_min_bounds = np.array([5, 10, 10, 5, 3, 1.2, 0.5, 0.2, 0.1, 0.1, 0.1]) * YEAR
var_max_bounds = np.array([50, 100, 100, 50, 30, 12, 5, 2, 1, 1, 1.]) * YEAR
var_bounds = (var_min_bounds, var_max_bounds)

# prob = pygmo.problem(pygmo.unconstrain(InitialSequenceProblem(MU, r0, sequence, 200*YEAR, var_bounds), method='ignore_o'))
prob = pygmo.problem(InitialSequenceProblem(MU, r0, sequence, 200*YEAR, var_bounds))
prob
# prob = InitialSequenceProblem(MU, r0, sequence, 200*YEAR, var_bounds)
# prob.extract(InitialSequenceProblem).fitness([1,0.5,1,1,0.5,1,1,0,1,1,1])

Problem name: <class '__main__.InitialSequenceProblem'>
	C++ class name: pybind11::object

	Global dimension:			11
	Integer dimension:			0
	Fitness dimension:			23
	Number of objectives:			1
	Equality constraints dimension:		12
	Inequality constraints dimension:	10
	Tolerances on constraints: [0, 0, 0, 0, 0, ... ]
	Lower bounds: [0, 0, 0, 0, 0, ... ]
	Upper bounds: [1, 1, 1, 1, 1, ... ]
	Has batch fitness evaluation: false

	Has gradient: false
	User implemented gradient sparsity: false
	Has hessians: false
	User implemented hessians sparsity: false

	Fitness evaluations: 0

	Thread safety: none

In [9]:
prob.fitness([1,0.5,1,1,0.5,1,1,0,1,1,1])

array([1.11082752e+10, 1.11082752e+10, 1.11082752e+10, 1.11082752e+10,
       1.11082752e+10, 1.11082752e+10, 1.11082752e+10, 1.11082752e+10,
       1.11082752e+10, 1.11082752e+10, 1.11082752e+10, 1.11082752e+10,
       1.11082752e+10, 1.11082752e+10, 1.11082752e+10, 1.11082752e+10,
       1.11082752e+10, 1.11082752e+10, 1.11082752e+10, 1.11082752e+10,
       1.11082752e+10, 1.11082752e+10, 1.11082752e+10])

In [65]:
algo = pygmo.algorithm(pygmo.gaco(gen=100))
algo.set_verbosity(1)

# 3. Create a population (a larger population size is generally better for global search)
pop = pygmo.population(prob=prob, size=64)

# 4. Evolve
pop = algo.evolve(pop)

# 5. Retrieve the best solution
champion_x = pop.champion_x
champion_f = pop.champion_f

          0        3.38752      0.0389875
     83           5248    1.11083e+09             63              0        2.24201      0.0321922
     84           5312    1.11083e+09             63              0        2.66718      0.0268799
     85           5376    1.11083e+09             63              0        3.55809      0.0254389
     86           5440    1.11083e+09             63              0        3.57988      0.0250163
     87           5504    1.11083e+09             63              0        2.90196       0.047449
     88           5568    1.11083e+09             63              0        3.57499      0.0337571
     89           5632    1.11083e+09             63              0        3.41104      0.0281212
     90           5696    1.11083e+09             63              0        1.64838      0.0232495
     91           5760    1.11083e+09             63              0         1.9866      0.0231761
     92           5824    1.11083e+09             63              0        3

In [66]:
champion_x = pop.champion_x
champion_f = pop.champion_f

In [67]:
champion_x

array([0.74993545, 0.74993545, 0.74993545, 0.74993545, 0.74993545,
       0.74993545, 0.74993545, 0.74993545, 0.74993545, 0.74993545,
       0.74993545])

In [68]:
champion_f

array([8.60826799e+09, 8.60826799e+09, 8.60826799e+09, 8.60826799e+09,
       8.60826799e+09, 8.60826799e+09, 8.60826799e+09, 8.60826799e+09,
       8.60826799e+09, 8.60826799e+09, 8.60826799e+09, 8.60826799e+09,
       8.60826799e+09, 8.60826799e+09, 8.60826799e+09, 8.60826799e+09,
       8.60826799e+09, 8.60826799e+09, 8.60826799e+09, 8.60826799e+09,
       8.60826799e+09, 8.60826799e+09, 8.60826799e+09])

In [28]:
prob.extract().inner_problem.fitness(champion_x)

array([9.95472082e+09, 9.95472082e+09, 9.95472082e+09, 9.95472082e+09,
       9.95472082e+09, 9.95472082e+09, 9.95472082e+09, 9.95472082e+09,
       9.95472082e+09, 9.95472082e+09, 9.95472082e+09, 9.95472082e+09,
       9.95472082e+09, 9.95472082e+09, 9.95472082e+09, 9.95472082e+09,
       9.95472082e+09, 9.95472082e+09, 9.95472082e+09, 9.95472082e+09,
       9.95472082e+09, 9.95472082e+09, 9.95472082e+09])

In [31]:
prob.get_bounds()

(array([1.57788e+08, 1.57788e+08, 1.57788e+08, 1.57788e+08, 1.57788e+08,
        1.57788e+08, 1.57788e+08, 1.57788e+08, 1.57788e+08, 1.57788e+08,
        1.57788e+08]),
 array([1.57788e+09, 1.57788e+09, 1.57788e+09, 1.57788e+09, 1.57788e+09,
        1.57788e+09, 1.57788e+09, 1.57788e+09, 1.57788e+09, 1.57788e+09,
        1.57788e+09]))